# Desafio Final da Trilha — Travel Planner com n8n + MCP

Estende o agente do **M1A7** (`m1a7_travel_planner.ipynb`) com **duas ferramentas externas**:

| Etapa | Ferramenta | Tipo | O que faz |
|------|------------|------|-----------|
| 1 | `clima_e_checklist` | **Webhook n8n** | manda o roteiro -> n8n consulta Open-Meteo -> devolve previsao + checklist de mala |
| 2 | `poi_find` | **Servidor MCP (FastMCP)** | dado lat/lon/categoria/raio, lista POIs reais (Overpass/OSM ou OpenTripMap) |

Mantidas do M1A7: `calcular_orcamento` (Python puro) e `preco_passagens` (busca web Tavily).

**Fluxo final do agente** (o LLM decide chamar as ferramentas no ciclo ReAct):
> Usuario: *"Planeje 3 dias em Lisboa e diga o que levar."*
> 1. agente rascunha destinos/datas → 2. `poi_find` p/ atracoes por dia → 3. `clima_e_checklist` (n8n) p/ clima+mala → 4. `calcular_orcamento` → 5. compila o plano final.

O grafo LangGraph e o mesmo do M1A7 (`query → search → agent ⇄ tools → feedback`, com human-in-the-loop e ate 3 revisoes); so trocamos o conjunto de ferramentas e tornamos os nos **assincronos** (preciso p/ as tools MCP).

## 0. Pre-requisitos

1. **Deps** (ja no `pyproject.toml`): `uv sync` na raiz do repo.
2. **`.env`** com `GOOGLE_API_KEY` (ou `OPENAI_API_KEY`) + `TAVILY_API_KEY` + `N8N_WEBHOOK_URL`. `OPENTRIPMAP_API_KEY` e opcional.
3. **n8n no ar** (Etapa 1):
   ```bash
   cd desafio-final && docker compose up -d      # (ou: podman compose up -d)
   ```
   Abra http://localhost:5678, importe `desafio-final/n8n/weather_packing_workflow.json`, **ative** o workflow.
4. **Servidor MCP** (Etapa 2): nao precisa subir nada — o cliente abaixo **spawna** `desafio-final/mcp_servers/poi_server.py` via stdio.

## 1. Setup do ambiente e provider

Igual ao M1A7 (dual provider Gemini/OpenAI, `temperature=0`), + descoberta da raiz do repo e da URL do webhook n8n.

In [ ]:
import os, sys, json
from typing import TypedDict, List, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from IPython.display import Image, display

load_dotenv()

# raiz do repo: sobe a partir do CWD ate achar a pasta "desafio-final"
REPO_ROOT = os.getcwd()
while REPO_ROOT != os.path.dirname(REPO_ROOT) and not os.path.isdir(os.path.join(REPO_ROOT, "desafio-final")):
    REPO_ROOT = os.path.dirname(REPO_ROOT)

PROVIDER = os.getenv("LLM_PROVIDER", "gemini").lower()
OPENAI_MODEL, GEMINI_MODEL = "gpt-4o-mini", "gemini-2.5-flash"

assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY ausente no .env (https://tavily.com)"
if PROVIDER == "openai":
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY ausente no .env"
    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(model=OPENAI_MODEL, temperature=0)
elif PROVIDER == "gemini":
    assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY ausente no .env"
    from langchain_google_genai import ChatGoogleGenerativeAI
    model = ChatGoogleGenerativeAI(model=GEMINI_MODEL, temperature=0)
else:
    raise ValueError(f"LLM_PROVIDER desconhecido: {PROVIDER}")

N8N_WEBHOOK_URL = os.getenv("N8N_WEBHOOK_URL", "http://localhost:5678/webhook/weather-packing")
print(f"Provider: {PROVIDER} | repo: {REPO_ROOT}")
print(f"n8n webhook: {N8N_WEBHOOK_URL}")

## 2. Ferramentas locais (Python + Tavily + n8n)

- `calcular_orcamento` — matematica pura (M1A7).
- `preco_passagens` — busca web via Tavily (M1A7).
- **`clima_e_checklist` (Etapa 1, NOVA)** — `@tool` **assincrona** que faz `POST` no webhook do n8n com o roteiro e devolve o JSON `{itinerary_weather, checklists}`. Trata o n8n fora do ar com mensagem clara (o agente continua).

`ParadaViagem` da um schema forte ao argumento `itinerary` (cidade + data; coordenadas opcionais — o n8n geocodifica se faltarem).

In [ ]:
import httpx
from langchain_core.tools import tool
from langchain_tavily import TavilySearch

search_tool = TavilySearch(max_results=3)

def _busca(query: str) -> str:
    """Busca Tavily -> trechos concatenados."""
    try:
        res = search_tool.invoke(query)
        results = (res.get("results") if isinstance(res, dict) else res) or []
        trechos = [r.get("content", "") for r in results]
        return "\n\n".join(t for t in trechos if t) or "Sem resultados."
    except Exception as e:
        return f"Erro na busca: {e}"

@tool
def calcular_orcamento(dias: int, diaria: float, custo_passagem: float = 0.0, extras: float = 0.0) -> str:
    """Calcula o orcamento total de uma viagem (diarias + passagem + extras)."""
    hospedagem = dias * diaria
    total = hospedagem + custo_passagem + extras
    return (f"Orcamento p/ {dias} dias:\n- Diarias ({dias} x {diaria:.2f}): {hospedagem:.2f}\n"
            f"- Passagem: {custo_passagem:.2f}\n- Extras: {extras:.2f}\n- TOTAL: {total:.2f}")

@tool
def preco_passagens(destino: str, origem: str = "Sao Paulo") -> str:
    """Busca precos estimados de passagens aereas de uma origem ate o destino (web)."""
    return _busca(f"preco passagem aerea de {origem} para {destino}")

class ParadaViagem(BaseModel):
    """Uma parada do roteiro enviada ao n8n."""
    city: str = Field(description="Cidade/destino, ex.: 'Lisboa'")
    date: Optional[str] = Field(default=None, description="Data YYYY-MM-DD (opcional)")
    lat: Optional[float] = Field(default=None, description="Latitude (opcional; n8n geocodifica se faltar)")
    lon: Optional[float] = Field(default=None, description="Longitude (opcional)")

@tool
async def clima_e_checklist(itinerary: List[ParadaViagem]) -> str:
    """Previsao do tempo + checklist de bagagem para cada parada do roteiro (via automacao n8n).

    Use SEMPRE que o usuario quiser saber o clima e/ou O QUE LEVAR NA MALA.
    Passe uma lista de paradas: [{city, date, lat?, lon?}]."""
    payload = {"itinerary": [p.model_dump(exclude_none=True) for p in itinerary]}
    try:
        async with httpx.AsyncClient(timeout=60) as client:
            resp = await client.post(N8N_WEBHOOK_URL, json=payload)
            resp.raise_for_status()
            return json.dumps(resp.json(), ensure_ascii=False)
    except httpx.HTTPStatusError as e:
        return (f"n8n respondeu HTTP {e.response.status_code}. O workflow esta ATIVO? "
                f"Em modo teste use a URL /webhook-test/. URL atual: {N8N_WEBHOOK_URL}")
    except httpx.HTTPError as e:
        return (f"Sem resposta do n8n em {N8N_WEBHOOK_URL} ({type(e).__name__}). "
                f"Suba o container (docker compose up -d) e ative o workflow.")

## 3. Ferramenta MCP (Etapa 2): carregando `poi_find` via FastMCP

O `MultiServerMCPClient` **spawna** o servidor `poi_server.py` por **stdio** e expoe suas tools como `BaseTool` do LangChain — entram no `bind_tools` como qualquer outra.

Dois detalhes que importam:
- **`command=sys.executable`**: usa o mesmo Python do venv (onde o `fastmcp` esta instalado).
- **`env=dict(os.environ)`**: o transporte stdio do MCP usa um env **minimo** por padrao; propagar o `os.environ` garante `PATH`, certificados TLS/proxy e a `OPENTRIPMAP_API_KEY` dentro do subprocesso.

Se o MCP falhar, o agente segue sem `poi_find` (degrada com elegancia).

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

POI_SERVER = os.path.join(REPO_ROOT, "desafio-final", "mcp_servers", "poi_server.py")
mcp_client = MultiServerMCPClient({
    "poi": {
        "transport": "stdio",
        "command": sys.executable,        # mesmo Python do venv
        "args": [POI_SERVER],
        "cwd": REPO_ROOT,
        "env": dict(os.environ),          # propaga PATH/CA/proxy/keys p/ o subprocesso
    }
})
try:
    mcp_tools = await mcp_client.get_tools()
    print("Tools MCP carregadas:", [t.name for t in mcp_tools])
except Exception as e:
    print(f"MCP indisponivel ({e}); seguindo sem poi_find.")
    mcp_tools = []

tools = [calcular_orcamento, preco_passagens, clima_e_checklist, *mcp_tools]
known_actions = {t.name: t for t in tools}
model_tools = model.bind_tools(tools)
print("Ferramentas do agente:", list(known_actions))

## 4. Estado e prompts

`AgentState` e `Queries` iguais ao M1A7. O `TOOLS_NOTE` foi reescrito para orientar o LLM a usar as ferramentas novas (`poi_find` p/ atracoes, `clima_e_checklist` p/ clima+mala).

In [ ]:
class AgentState(TypedDict):
    task: str
    queries: List[str]
    draft: List[str]
    messages: list
    result: str
    user_feedback: str
    revision_count: int

class Queries(BaseModel):
    queries: List[str]

In [ ]:
QUERY_MAKER_PROMPT = """
Voce e um especialista em planejamento de viagens.
Recebera um destino e interesses e deve gerar 5-8 queries especificas para buscar:
atividades ligadas aos interesses, locais, melhor epoca e infos praticas.
Retorne apenas as queries, uma por linha, sem numeracao.
"""

TOOLS_NOTE = """
Voce tem ferramentas e DEVE usa-las quando o pedido exigir (NAO invente dados):
- poi_find(lat, lon, radius_m, kinds, limit): atracoes/POIs REAIS por coordenadas
  (kinds: museums, cultural, restaurants, parks, viewpoints, historic, churches...).
  Use para montar o que visitar em cada dia/bairro. Informe coordenadas do destino.
- clima_e_checklist(itinerary): previsao do tempo + checklist de bagagem (n8n).
  Use SEMPRE que pedirem clima ou O QUE LEVAR. Passe [{city, date, lat?, lon?}].
- preco_passagens(destino, origem): preco estimado de passagens (web).
- calcular_orcamento(dias, diaria, custo_passagem, extras): orcamento total.
Fluxo tipico: rascunhe destinos/datas -> poi_find -> clima_e_checklist -> calcular_orcamento.
"""

RESULT_PROMPT = """
Voce e um especialista em planejamento de viagens.
Com base nas informacoes coletadas, gere um planejamento completo, amigavel e envolvente.
Estruture com: 1. INTRODUCAO  2. ATIVIDADES (use POIs reais do poi_find)  3. LOCAIS
4. DICAS PRATICAS (clima + checklist do clima_e_checklist; orcamento; passagens)  5. CRONOGRAMA por dia.
""" + TOOLS_NOTE + """
Informacoes coletadas:
{draft}

Interesses do usuario: {task}

Gere o planejamento completo:
"""

REVISION_PROMPT = """
Voce e um especialista em planejamento de viagens.
Revise o planejamento abaixo com base no feedback, usando as informacoes coletadas e as ferramentas.

PLANEJAMENTO ATUAL:
{current_result}

FEEDBACK DO USUARIO:
{user_feedback}

INTERESSES ORIGINAIS:
{task}

INFORMACOES COLETADAS:
{draft}
""" + TOOLS_NOTE + """
Mantenha a estrutura organizada. Gere o planejamento revisado:
"""

## 5. Nos do grafo (assincronos)

Mesma logica do M1A7, mas `async` para suportar as tools MCP:
- `query_node`/`agent_node`: usam `await model...ainvoke(...)`.
- `tools_node`: `await tool.ainvoke(...)` e **normaliza** o retorno MCP (lista de content-blocks `{type, text}`) para texto antes de virar `ToolMessage`.
- `search_node`/`feedback_node`: sincronos (Tavily / `input()`).

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

async def query_node(state: AgentState):
    msgs = [SystemMessage(content=QUERY_MAKER_PROMPT), HumanMessage(content=state["task"])]
    response = await model.with_structured_output(Queries).ainvoke(msgs)
    return {"queries": response.queries}

def search_node(state: AgentState):
    draft = state.get("draft") or []
    for q in state["queries"]:
        draft.append(_busca(q))
    return {"draft": draft}

async def agent_node(state: AgentState):
    msgs = state.get("messages") or []
    last = msgs[-1] if msgs else None
    fresh = (not msgs) or isinstance(last, AIMessage)   # turno novo (inicio ou revisao)
    if fresh:
        draft = "\n\n".join(state.get("draft") or [])
        fb = state.get("user_feedback")
        if fb and fb not in ("", "Aprovado", "Sair"):
            sys_p = REVISION_PROMPT.format(current_result=state.get("result", ""),
                                           user_feedback=fb, task=state["task"], draft=draft)
        else:
            sys_p = RESULT_PROMPT.format(draft=draft, task=state["task"])
        msgs = [SystemMessage(content=sys_p), HumanMessage(content=state["task"])]
    response = await model_tools.ainvoke(msgs)
    msgs = msgs + [response]
    out = {"messages": msgs}
    if not response.tool_calls:
        out["result"] = response.content
    return out

async def tools_node(state: AgentState):
    last = state["messages"][-1]
    results = []
    for call in last.tool_calls:
        print(f"  \U0001f527 tool: {call['name']}({call['args']})")
        obs = await known_actions[call["name"]].ainvoke(call["args"])
        if isinstance(obs, list):   # content-blocks do MCP -> texto
            obs = "\n".join(b.get("text", "") if isinstance(b, dict) else str(b) for b in obs)
        results.append(ToolMessage(tool_call_id=call["id"], name=call["name"], content=str(obs)))
    return {"messages": state["messages"] + results}

def feedback_node(state: AgentState):
    print("\n" + "=" * 60 + "\n\U0001f3af SEU PLANO DE VIAGEM:\n" + "=" * 60)
    print(state["result"])
    print("=" * 60)
    print("\n\U0001f4ac Alteracoes? Digite sugestoes, 'ok' p/ finalizar, ou 'sair'.")
    feedback = input("\nSua resposta: ").strip()
    rc = state.get("revision_count", 0)
    if feedback.lower() in ("ok", "perfeito", "finalizar", "done"):
        return {"user_feedback": "Aprovado", "revision_count": rc, "messages": []}
    if feedback.lower() in ("sair", "exit", "quit"):
        return {"user_feedback": "Sair", "revision_count": rc, "messages": []}
    return {"user_feedback": feedback, "revision_count": rc + 1, "messages": []}

def route_after_agent(state: AgentState):
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", None) else "feedback"

def should_continue(state: AgentState):
    if state["user_feedback"] in ("Aprovado", "Sair"):
        return "end"
    if state.get("revision_count", 0) >= 3:
        print("\n⚠️  Limite de 3 revisoes atingido.")
        return "end"
    return "revise"

## 6. Construcao do grafo

In [ ]:
builder = StateGraph(AgentState)
builder.add_node("query", query_node)
builder.add_node("search", search_node)
builder.add_node("agent", agent_node)
builder.add_node("tools", tools_node)
builder.add_node("feedback", feedback_node)

builder.add_edge(START, "query")
builder.add_edge("query", "search")
builder.add_edge("search", "agent")
builder.add_conditional_edges("agent", route_after_agent, {"tools": "tools", "feedback": "feedback"})
builder.add_edge("tools", "agent")
builder.add_conditional_edges("feedback", should_continue, {"revise": "agent", "end": END})

graph = builder.compile(checkpointer=InMemorySaver())

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"(viz indisponivel: {e})")
    print(graph.get_graph().draw_ascii())

## 7. Smoke test das ferramentas novas

Testa as duas tools isoladas antes do grafo. `poi_find` precisa do server MCP (carregado na secao 3); `clima_e_checklist` precisa do n8n no ar.

In [ ]:
if mcp_tools:
    print("=== poi_find (MCP) — Lisboa, museus + miradouros ===")
    print(await known_actions["poi_find"].ainvoke(
        {"lat": 38.7223, "lon": -9.1393, "radius_m": 1000, "kinds": "museums,viewpoints", "limit": 5}))

print("\n=== clima_e_checklist (n8n) ===")
print(await clima_e_checklist.ainvoke({"itinerary": [{"city": "Lisboa", "date": "2026-06-15"}]}))

print("\n=== calcular_orcamento (Python) ===")
print(calcular_orcamento.invoke({"dias": 3, "diaria": 250, "custo_passagem": 2800, "extras": 300}))

## 8. Teste end-to-end (human-in-the-loop)

Observe os logs `\U0001f527 tool: ...` — prova de que o agente esta chamando `poi_find` (MCP) e `clima_e_checklist` (n8n). Quando o plano aparecer, digite uma sugestao para revisar, ou `ok` para finalizar.

In [ ]:
async def run_travel_planner(task: str, thread_id: str = "1"):
    print("\U0001f680 Travel Planner+ (n8n + MCP)\n" + "=" * 50)
    thread = {"configurable": {"thread_id": thread_id}}
    init = {"task": task, "queries": [], "draft": [], "messages": [],
            "result": "", "user_feedback": "", "revision_count": 0}
    async for step in graph.astream(init, thread):
        if "query" in step:
            print(f"\U0001f50d {len(step['query']['queries'])} queries geradas")
        elif "search" in step:
            print(f"\U0001f4da {len(step['search']['draft'])} fontes coletadas")
        elif "agent" in step and step["agent"].get("result"):
            print("✅ Plano gerado/revisado!")
        elif "feedback" in step:
            fb = step["feedback"]["user_feedback"]
            if fb == "Aprovado":
                print("\n\U0001f389 Plano aprovado!")
            elif fb == "Sair":
                print("\n\U0001f44b Ate a proxima!")
            else:
                print(f"\U0001f504 Revisao #{step['feedback']['revision_count']} solicitada...")

await run_travel_planner(
    "Planeje 3 dias em Lisboa, com atracoes por dia, o que levar na mala e um orcamento estimado."
)

## 9. Como isto atende ao desafio

| Etapa do enunciado | Onde esta |
|---|---|
| **n8n "Weather & Packing Checklist" (webhook)** | tool `clima_e_checklist` → `desafio-final/n8n/weather_packing_workflow.json` (Webhook → Open-Meteo → checklist → Respond) |
| **Servidor MCP FastMCP "POI Finder"** | tool `poi_find` → `desafio-final/mcp_servers/poi_server.py` (`@mcp.tool`, contrato forte, Overpass/OSM ou OpenTripMap) |
| **Agente chama as duas no ciclo de raciocinio** | `model.bind_tools([... , clima_e_checklist, poi_find])` + grafo `agent ⇄ tools` |

O fluxo pedido (rascunho → `poi_find` por dia → `clima_e_checklist` p/ clima+mala → orcamento → resposta estruturada) emerge naturalmente do tool-calling. Detalhes de setup no `desafio-final/README.md`.